# Exp 2 - Latency and Jitter Measurement using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Measure latency and jitter for a packet stream and explain why timing variation matters in autonomous communication.

- **Latency:** end-to-end delay for one packet.
- **Jitter:** variation in latency across packets.
- **Tail latency:** high-percentile latency such as P95 or P99, useful for identifying rare but important timing spikes.

## Textbook Notes and Case Studies

### 1. Textbook Background

Latency is the time taken for a message, packet, or event response to travel from source to destination. Jitter is the variation in latency across repeated transmissions. In real-time networks, jitter can be as important as average latency because control loops depend on predictable timing.

For autonomous systems, average performance alone is not enough. A network path with 5 ms average latency but occasional 100 ms spikes may be worse for control than a path with a steady 15 ms latency. The correct engineering question is: what is the worst-case or high-percentile delay, and does it remain below the deadline?

### 2. Architecture Notes

```
Sender Timestamp -> Network / Queue / Link -> Receiver Timestamp
        |                                      |
        +-------------- Delay Log ------------+
                         |
                         v
             Mean / Min / Max / Jitter Analysis
```

The experiment simulates timestamped transmissions. Each packet has a send time and receive time. The difference is the latency. Jitter is computed from the change in latency between packets or from the standard deviation of the latency series.

### 3. Important Formulas

One-way latency:

```
latency = receive_time - send_time
```

Average latency:

```
mean_latency = sum(latencies) / n
```

Packet-to-packet jitter:

```
jitter_i = abs(latency_i - latency_(i-1))
```

Mean jitter:

```
mean_jitter = sum(jitter_values) / number_of_jitter_values
```

Deadline violation rate:

```
violation_rate = packets_exceeding_deadline / total_packets
```

### 4. Classroom Case Studies

Case Study A - Cooperative Adaptive Cruise Control:
Vehicles exchange acceleration and spacing information. If latency is low and stable, the following vehicle can react smoothly. If jitter is high, the controller may receive uneven updates and produce unstable acceleration commands.

Case Study B - Remote Driving:
A remote operator receives camera frames and sends steering commands. Average latency hides the problem: one delayed command during a turn can be more damaging than many low-latency packets during straight driving.

Case Study C - Platooning:
In vehicle platoons, the lead vehicle shares braking intent. Jitter affects the spacing policy because the follower must reserve enough safety margin for delayed or irregular messages.

### 5. Analysis Checklist

Report minimum, maximum, mean, and jitter. Never conclude only from the mean. A strong lab answer explains whether the timing distribution is narrow or unstable. If a deadline is given, include the number and percentage of deadline violations.

### 6. Source Notes

- Python's high-resolution clocks are documented in the official Python `time` module documentation: https://docs.python.org/3/library/time.html
- IEEE 802.1 Time-Sensitive Networking standards are relevant background for bounded-latency Ethernet: https://1.ieee802.org/tsn/


## Architecture

```text
Packet Generator
  |-- packet id
  |-- send timestamp
          |
          v
Network Delay Model
  |-- base delay
  |-- random variation
  |-- occasional burst delay
          |
          v
Receiver
  |-- receive timestamp
          |
          v
Timing Analyzer
  |-- latency per packet
  |-- jitter between adjacent packets
  |-- mean, standard deviation, P95, P99
```

The model is deterministic because it uses fixed random seeds. That makes the result reproducible across notebook runs.

## Formulas and Required Theory

\[
\text{latency}_i = T_{receive,i} - T_{send,i}
\]

\[
\text{jitter}_i = |\text{latency}_i - \text{latency}_{i-1}|
\]

\[
\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i
\]

Percentiles are computed by sorting latency values and interpolating at the requested rank. P99 is important because the mean can hide rare delay spikes that still break a control deadline.

## In-Lab Method

1. Generate 60 packets.
2. Assign each packet a send time and simulated network delay.
3. Compute latency as receive time minus send time.
4. Compute jitter as the absolute difference between consecutive latency values.
5. Report mean, standard deviation, P95, P99, and average jitter.

In [1]:
import math
import random
import statistics

def percentile(values, pct):
    values = sorted(values)
    pos = (len(values) - 1) * pct
    lo, hi = math.floor(pos), math.ceil(pos)
    return values[lo] if lo == hi else values[lo] * (hi - pos) + values[hi] * (pos - lo)

rng = random.Random(341402)
latencies = []
for packet_id in range(60):
    delay = 8.0 + rng.uniform(-1.4, 1.4) + rng.expovariate(1 / 1.8)
    if packet_id % 17 == 0:
        delay += rng.uniform(0, 4.0)
    latencies.append(delay)
jitters = [abs(latencies[i] - latencies[i - 1]) for i in range(1, len(latencies))]

print("EXP 2 - IN-LAB RESULT")
print(f"Packets         : {len(latencies)}")
print(f"Mean latency    : {statistics.mean(latencies):.2f} ms")
print(f"Std deviation   : {statistics.stdev(latencies):.2f} ms")
print(f"P95 latency     : {percentile(latencies, 0.95):.2f} ms")
print(f"P99 latency     : {percentile(latencies, 0.99):.2f} ms")
print(f"Average jitter  : {statistics.mean(jitters):.2f} ms")

EXP 2 - IN-LAB RESULT
Packets         : 60
Mean latency    : 10.16 ms
Std deviation   : 2.33 ms
P95 latency     : 14.35 ms
P99 latency     : 16.33 ms
Average jitter  : 2.56 ms


## Post-Lab Method

The post-lab cell injects 0-20 ms random delay to emulate congestion. It compares baseline and congested traffic, then evaluates a moving-average jitter buffer.

Trade-off:

- larger buffer window -> lower jitter
- larger buffer window -> more waiting delay

For control traffic, smoothing is useful only if the added delay remains within the control-loop deadline.

In [2]:
import math
import random
import statistics

def percentile(values, pct):
    values = sorted(values)
    pos = (len(values) - 1) * pct
    lo, hi = math.floor(pos), math.ceil(pos)
    return values[lo] if lo == hi else values[lo] * (hi - pos) + values[hi] * (pos - lo)

def stats(extra_delay=False, seed=0):
    rng = random.Random(seed)
    lat = [8 + rng.uniform(-1.4, 1.4) + rng.expovariate(1 / 1.8) + (rng.uniform(0, 20) if extra_delay else 0) for _ in range(60)]
    jit = [abs(lat[i] - lat[i - 1]) for i in range(1, len(lat))]
    return statistics.mean(lat), statistics.stdev(lat), percentile(lat, 0.95), percentile(lat, 0.99), statistics.mean(jit)

print("EXP 2 - POST-LAB CONGESTION COMPARISON")
print(f"{'Condition':22} {'Mean':>8} {'Std':>8} {'P95':>8} {'P99':>8} {'Jitter':>8}")
for name, extra, seed in [("Baseline", False, 341402), ("0-20 ms delay", True, 341403)]:
    mean, stdev, p95, p99, jitter = stats(extra, seed)
    print(f"{name:22} {mean:8.2f} {stdev:8.2f} {p95:8.2f} {p99:8.2f} {jitter:8.2f}")

EXP 2 - POST-LAB CONGESTION COMPARISON
Condition                  Mean      Std      P95      P99   Jitter
Baseline                  10.03     2.29    13.77    16.00     2.20
0-20 ms delay             20.40     6.79    30.50    31.77     7.56


## What to Write in the Lab Record

- Include the baseline timing table.
- Include the congestion comparison table.
- Explain why P95/P99 are more meaningful than the mean for safety-critical messages.
- State the moving-average window that gives the best jitter-delay trade-off.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html